# Module 12 — Black-Box Optimisation

This notebook analyses the eight initial datasets and proposes one **new** input per function using a Gaussian Process and Expected Improvement. All tasks are maximisation. Candidate points are **model-guided suggestions, not proven optima**.

Place the course ZIP file `Initial_data_points_starter(1).zip` alongside this notebook. Outputs and charts are generated by running all cells. The domain `[0,1]^d` is assumed; verify against the course rules before submitting.

In [ ]:
import io, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, qmc
from scipy.spatial.distance import cdist
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel
from pathlib import Path

ZIP_PATH = Path("Initial_data_points_starter(1).zip")
assert ZIP_PATH.exists(), f"Missing {ZIP_PATH}"
Path("graphs").mkdir(exist_ok=True)

In [ ]:
datasets = {}
with zipfile.ZipFile(ZIP_PATH) as archive:
    for i in range(1, 9):
        folder = f"initial_data/function_{i}/"
        X = np.load(io.BytesIO(archive.read(folder + "initial_inputs.npy")))
        y = np.load(io.BytesIO(archive.read(folder + "initial_outputs.npy"))).ravel()
        assert X.shape[0] == y.shape[0]
        datasets[i] = (X, y)
        print(f"Function {i}: X={X.shape}, y={y.shape}, best={y.max():.6f}")

## Initial data visualisations

For 2D functions, the scatter plots show input coordinates coloured by observed output. For every function, an output-by-observation plot is generated. The dashed line indicates the highest observed value, not the true maximum.

In [ ]:
for i, (X, y) in datasets.items():
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(np.arange(1, len(y)+1), y)
    ax.axhline(y.max(), linestyle="--", color="gray", label="Best observed")
    ax.set(xlabel="Observation number", ylabel="Observed output", title=f"Function {i}: observed outputs")
    ax.legend()
    fig.tight_layout()
    fig.savefig(f"graphs/function_{i}_outputs.png", dpi=160)
    plt.show()
    if X.shape[1] == 2:
        fig, ax = plt.subplots(figsize=(7, 5))
        scatter = ax.scatter(X[:, 0], X[:, 1], c=y, s=90, edgecolor="black")
        fig.colorbar(scatter, ax=ax, label="Observed output")
        ax.set(xlabel="Input 1", ylabel="Input 2", title=f"Function {i}: initial inputs")
        fig.tight_layout()
        fig.savefig(f"graphs/function_{i}_inputs.png", dpi=160)
        plt.show()

## Bayesian optimisation: GP + Expected Improvement

Each output vector is standardised before GP fitting. We use a Matérn 5/2 covariance kernel and a small learned noise term. Sobol candidate points cover the unit hypercube. Expected Improvement (EI) balances high predicted outputs and predictive uncertainty. Candidate points too close to existing inputs are excluded. For higher-dimensional functions, GP estimates can be uncertain because data are sparse.

In [ ]:
def expected_improvement(mu, sigma, best, xi=0.01):
    improvement = mu - best - xi
    z = improvement / np.maximum(sigma, 1e-12)
    return improvement * norm.cdf(z) + sigma * norm.pdf(z)

results = []
for i, (X, y) in datasets.items():
    d = X.shape[1]
    y_mean, y_scale = y.mean(), max(y.std(), 1e-8)
    y_scaled = (y-y_mean)/y_scale
    kernel = (ConstantKernel(1.0, (0.01, 100)) *
              Matern(length_scale=np.full(d, 0.25), length_scale_bounds=(0.04, 5), nu=2.5) +
              WhiteKernel(noise_level=0.01, noise_level_bounds=(1e-6, 0.5)))
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=2, random_state=42)
    gp.fit(X, y_scaled)
    candidates = qmc.Sobol(d, scramble=True, seed=123+i).random_base2(m=13)
    mean, sigma = gp.predict(candidates, return_std=True)
    ei = expected_improvement(mean, sigma, y_scaled.max())
    ei[cdist(candidates, X).min(axis=1) < 0.025] = 0
    j = int(np.argmax(ei))
    k = int(np.argmax(y))
    point = candidates[j]
    results.append({"function": i, "dimensions": d, "observations": len(y),
                    "best_observation": k+1, "best_output": float(y[k]),
                    "best_input": "-".join(f"{v:.6f}" for v in X[k]),
                    "proposed_query": "-".join(f"{v:.6f}" for v in point),
                    "predicted_mean": float(mean[j]*y_scale+y_mean),
                    "predicted_std": float(sigma[j]*y_scale),
                    "method": "Gaussian process Expected Improvement"})
results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv("module12_query_points.csv", index=False)

## Reflection prompts

- **Method:** Gaussian Process with Matérn kernel and Expected Improvement acquisition.
- **Exploration vs exploitation:** EI rewards both predicted improvement and uncertainty; in high dimensions predictions are less reliable.
- **What was learned:** Compare the new outputs with the initial best values and inspect whether predictions were accurate.
- **Next round:** Append the new input/output pairs to each dataset, refit the models and select new queries.

**Limitations:** The objective functions are unknown. The selected points are not guaranteed maxima. Model choices, input bounds and observed data quality influence the results. Avoid treating the predicted means as actual outcomes.